In [30]:
import joblib

SCALER_PATH = PROJECT_ROOT / "models" / "minmax_scaler.joblib"

joblib.dump(scaler, SCALER_PATH)

print("Scaler sauvegardé :", SCALER_PATH)
print("Fichier trouvé :", SCALER_PATH.exists())

Scaler sauvegardé : c:\Users\MSI\Desktop\fraud_detection_memoire\models\minmax_scaler.joblib
Fichier trouvé : True


## Sauvegarde du scaler

Le `MinMaxScaler` ajusté sur l'ensemble d'entraînement est sauvegardé afin de réutiliser exactement les mêmes paramètres de normalisation lors des étapes suivantes.

In [29]:
X_train_scaled.to_parquet(PROCESSED_DIR / "X_train.parquet")
X_val_scaled.to_parquet(PROCESSED_DIR / "X_val.parquet")
X_test_scaled.to_parquet(PROCESSED_DIR / "X_test.parquet")

y_train.to_frame(name="Class").to_parquet(PROCESSED_DIR / "y_train.parquet")
y_val.to_frame(name="Class").to_parquet(PROCESSED_DIR / "y_val.parquet")
y_test.to_frame(name="Class").to_parquet(PROCESSED_DIR / "y_test.parquet")

print("Données prétraitées sauvegardées avec succès.")

Données prétraitées sauvegardées avec succès.


In [28]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier de sauvegarde :", PROCESSED_DIR)

Dossier de sauvegarde : c:\Users\MSI\Desktop\fraud_detection_memoire\data\processed


## Sauvegarde des données prétraitées

Les ensembles d'entraînement, de validation et de test prétraités sont sauvegardés afin d'être réutilisés dans les étapes expérimentales suivantes.

In [23]:
print("Valeurs manquantes :")
print("Train      :", X_train_scaled.isna().sum().sum())
print("Validation :", X_val_scaled.isna().sum().sum())
print("Test       :", X_test_scaled.isna().sum().sum())

print("\nValeurs infinies :")
print("Train      :", np.isinf(X_train_scaled).sum().sum())
print("Validation :", np.isinf(X_val_scaled).sum().sum())
print("Test       :", np.isinf(X_test_scaled).sum().sum())

Valeurs manquantes :
Train      : 0
Validation : 0
Test       : 0

Valeurs infinies :
Train      : 0
Validation : 0
Test       : 0


In [24]:
diagnostic_val = pd.DataFrame({
    "Minimum Validation": X_val_scaled.min(),
    "Maximum Validation": X_val_scaled.max()
})

diagnostic_val[
    (diagnostic_val["Minimum Validation"] < 0) |
    (diagnostic_val["Maximum Validation"] > 1)
]

,Minimum Validation,Maximum Validation
V3,-0.340070,0.877371
V5,-1.220609,0.971743
V6,0.088977,2.091581
V7,0.246340,2.108218
V12,-0.004925,0.872151
V13,0.190663,1.246878
V15,0.056242,1.295613
V16,0.092374,1.484024
V17,0.102448,1.021226
V18,0.044985,1.023128


In [25]:
print("Train :")
print("Minimum :", X_train_scaled.min().min())
print("Maximum :", X_train_scaled.max().max())

print("\nValidation :")
print("Minimum :", X_val_scaled.min().min())
print("Maximum :", X_val_scaled.max().max())

print("\nTest :")
print("Minimum :", X_test_scaled.min().min())
print("Maximum :", X_test_scaled.max().max())

Train :
Minimum : 0.0
Maximum : 1.0

Validation :
Minimum : -1.220609143963074
Maximum : 2.108217998701164

Test :
Minimum : -0.15032439684441068
Maximum : 1.1252405869785513


In [26]:
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [27]:
scaler = MinMaxScaler() 
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)                                                                                                             

In [17]:
from sklearn.preprocessing import MinMaxScaler

## Normalisation des variables

Les variables explicatives sont normalisées à l'aide de `MinMaxScaler`. Les paramètres de normalisation sont estimés uniquement sur l'ensemble d'entraînement afin d'éviter toute fuite d'information.

In [16]:
X_train["Amount"] = np.log1p(X_train["Amount"])
X_val["Amount"] = np.log1p(X_val["Amount"])
X_test["Amount"] = np.log1p(X_test["Amount"])

## Transformation de la variable Amount

Une transformation logarithmique est appliquée à la variable `Amount` afin de réduire son asymétrie et l'influence des montants extrêmement élevés.

In [15]:
resume_split = pd.DataFrame({
    "Ensemble": ["Train", "Validation", "Test"],
    "Transactions totales": [
        len(X_train),
        len(X_val),
        len(X_test)
    ],
    "Transactions normales": [
        len(X_train) - y_train.sum(),
        len(X_val) - y_val.sum(),
        len(X_test) - y_test.sum()
    ],
    "Fraudes": [
        y_train.sum(),
        y_val.sum(),
        y_test.sum()
    ],
    "Taux normales (%)": [
        (1 - y_train.mean()) * 100,
        (1 - y_val.mean()) * 100,
        (1 - y_test.mean()) * 100
    ],
    "Taux de fraude (%)": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100
    ]
})

resume_split.style.format({
    "Taux normales (%)": "{:.4f}%",
    "Taux de fraude (%)": "{:.4f}%"
}).background_gradient(cmap="viridis")

,Ensemble,Transactions totales,Transactions normales,Fraudes,Taux normales (%),Taux de fraude (%)
0,Train,198608,198277,331,99.8333%,0.1667%
1,Validation,42559,42488,71,99.8332%,0.1668%
2,Test,42559,42488,71,99.8332%,0.1668%


In [13]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)

Validation : (42559, 30) (42559,)
Test       : (42559, 30) (42559,)


In [12]:
print("Train :", X_train.shape, y_train.shape)
print("Temp  :", X_temp.shape, y_temp.shape)

Train : (198608, 30) (198608,)
Temp  : (85118, 30) (85118,)


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)


In [10]:
from sklearn.model_selection import train_test_split

In [9]:
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

print("\nPremières colonnes de X :")
print(X.columns.tolist())

print("\nRépartition de y :")
print(y.value_counts())

Dimensions de X : (283726, 30)
Dimensions de y : (283726,)

Premières colonnes de X :
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

Répartition de y :
Class
0    283253
1       473
Name: count, dtype: int64


In [8]:
X = df_clean.drop(columns=["Class"]).copy()
y = df_clean["Class"].copy()

## Séparation des variables explicatives et de la variable cible

La variable `Class` constitue la cible à prédire, tandis que les autres colonnes sont utilisées comme variables explicatives.

In [7]:
print("Dimensions après suppression :", df_clean.shape)
print("Doublons restants :", df_clean.duplicated().sum())

print("\nRépartition des classes après suppression :")
print(df_clean["Class"].value_counts())

Dimensions après suppression : (283726, 31)
Doublons restants : 0

Répartition des classes après suppression :
Class
0    283253
1       473
Name: count, dtype: int64


In [6]:
df_clean = df.drop_duplicates().reset_index(drop=True)

In [5]:
duplicates_to_remove = df.duplicated(keep="first")

print("Lignes qui seront supprimées :", duplicates_to_remove.sum())
print("\nRépartition par classe :")
print(df.loc[duplicates_to_remove, "Class"].value_counts())

Lignes qui seront supprimées : 1081

Répartition par classe :
Class
0    1062
1      19
Name: count, dtype: int64


In [4]:
print("Dimensions avant suppression :", df.shape)
print("Doublons exacts :", df.duplicated().sum())

print("\nRépartition des classes avant suppression :")
print(df["Class"].value_counts())

Dimensions avant suppression : (284807, 31)
Doublons exacts : 1081

Répartition des classes avant suppression :
Class
0    284315
1       492
Name: count, dtype: int64


In [3]:
df = pd.read_csv(DATA_PATH)

print("Dimensions du dataset brut :", df.shape)

Dimensions du dataset brut : (284807, 31)


In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "creditcard.csv"

print("Dossier du projet :", PROJECT_ROOT)
print("Dataset brut :", DATA_PATH)
print("Fichier trouvé :", DATA_PATH.exists())

Dossier du projet : c:\Users\MSI\Desktop\fraud_detection_memoire
Dataset brut : c:\Users\MSI\Desktop\fraud_detection_memoire\data\raw\creditcard.csv
Fichier trouvé : True


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

# 02 — Prétraitement des données
## European Credit Card Fraud Detection

Objectif : préparer les données pour l'entraînement des modèles tout en évitant toute fuite d'information entre les ensembles d'entraînement, de validation et de test.